# 06 · Analysis

**Research question:** can machine learning reconstruct the SPY implied volatility surface more
accurately than SVI, and does any gain hold up in crisis periods?

Test set: every trading day from March 2020 to December 2022 (709 days); three masks per day
(random 20%, wings = outer 10% each side per expiry, one whole expiry). Errors are in
**volatility points** (1 = 0.01 of IV) and computed only on hidden quotes.

In [ ]:
import sys
sys.path.append('..')
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 160)

In [ ]:
import os, glob
from src.stats import compare_to_benchmark
from src.hedging import hedge_errors, hedge_summary

ORDER = ['mean', 'linear', 'svi', 'ssvi', 'xgb', 'xgb_lin', 'xgb_svi', 'gp', 'nn', 'nn_nopen']
LABEL = {'mean': 'Mean (floor)', 'linear': 'B0 Linear', 'svi': 'B1 SVI', 'ssvi': 'B2 SSVI',
         'xgb': 'M1a XGB direct', 'xgb_lin': 'M1b XGB on linear', 'xgb_svi': 'M2 XGB on SVI',
         'gp': 'M3 GP', 'nn': 'M4 NN + penalty', 'nn_nopen': "M4' NN no penalty"}
MODELS = [m for m in ORDER if os.path.exists(f'../results/{m}/scores.csv')]
scores = pd.concat([pd.read_csv(f'../results/{m}/scores.csv', parse_dates=['date'])
                    for m in MODELS], ignore_index=True)
scores = scores[scores['date'] >= '2020-03-01']
print(MODELS)
scores.groupby(['model', 'mask']).size().unstack()

## 1. Main result: accuracy by mask

In [ ]:
main = scores.pivot_table(index='model', columns='mask', values='rmse', aggfunc='mean')
main = main.loc[MODELS, ['random', 'expiry', 'wing']].rename(index=LABEL)
main.round(2)

Spread across days (5% / median / 95% quantiles of daily RMSE), as in Deep Smoothing.

In [ ]:
q = scores.groupby(['model', 'mask'])['rmse'].quantile([0.05, 0.5, 0.95]).unstack()
q.columns = ['q05', 'median', 'q95']
q.unstack('mask').loc[MODELS].rename(index=LABEL).round(2)

## 2. Does it hold up in a crisis? Accuracy by market regime

In [ ]:
for mask in ['random', 'expiry', 'wing']:
    t = scores[scores['mask'] == mask].pivot_table(index='model', columns='regime',
                                                   values='rmse', aggfunc='mean')
    t = t.loc[MODELS, ['covid_crash', 'recovery', 'calm_2021', 'bear_2022']].rename(index=LABEL)
    print(f'--- {mask} mask: mean daily RMSE (vol points)')
    display(t.round(2))

In [ ]:
SHOW = [m for m in ['linear', 'svi', 'ssvi', 'xgb_svi', 'gp', 'nn'] if m in MODELS]
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
for ax, mask in zip(axes, ['random', 'wing']):
    for m in SHOW:
        s = scores[(scores['model'] == m) & (scores['mask'] == mask)].set_index('date')['rmse']
        ax.plot(s.sort_index().rolling(21, min_periods=5).mean(), label=LABEL[m])
    ax.axvspan(pd.Timestamp('2020-02-20'), pd.Timestamp('2020-04-30'), color='grey', alpha=0.2)
    ax.set_title(f'{mask} mask — 21-day rolling mean RMSE'); ax.set_ylabel('vol points')
axes[0].legend(ncol=3, fontsize=8); plt.tight_layout(); plt.show()

## 3. Is the gain over SVI statistically significant?

Diebold–Mariano test on daily squared errors (Newey–West, 5 lags), and a 95% moving-block
bootstrap interval (10-day blocks) for the % improvement in mean RMSE over SVI.
Positive improvement = better than SVI.

In [ ]:
sig = compare_to_benchmark(scores, benchmark='svi')
sig['model'] = sig['model'].map(LABEL)
sig.round({'rmse_model': 2, 'rmse_svi': 2, 'improvement_%': 1, 'ci_low_%': 1, 'ci_high_%': 1,
           'dm_stat': 2, 'p_value': 4, 'win_rate_%': 1})

## 4. Where on the surface are the errors?

Regional RMSE: rows = standardised moneyness z = k/√T, columns = maturity buckets.

In [ ]:
import seaborn as sns
mask = 'wing'
pair = [m for m in ['svi', 'xgb_svi', 'nn'] if m in MODELS]
fig, axes = plt.subplots(1, len(pair), figsize=(6 * len(pair), 4.5))
for ax, m in zip(np.atleast_1d(axes), pair):
    tab = pd.read_csv(f'../results/{m}/regional_{mask}.csv', index_col=0)
    sns.heatmap(tab, annot=True, fmt='.1f', cmap='Reds', vmin=0, vmax=12, ax=ax, cbar=False)
    ax.set_title(f'{LABEL[m]} — {mask} mask')
plt.tight_layout(); plt.show()

## 5. Accuracy vs no-arbitrage

In [ ]:
arb = scores[scores['mask'] == 'random'].groupby('model')[
    ['rmse', 'butterfly_rate', 'calendar_rate', 'butterfly_size', 'calendar_size']].mean()
arb.loc[MODELS].rename(index=LABEL).round(4)

In [ ]:
a = arb.loc[[m for m in MODELS if m != 'mean']]
a['any_violation'] = a['butterfly_rate'] + a['calendar_rate']
plt.figure(figsize=(7, 5))
plt.scatter(a['any_violation'] * 100, a['rmse'])
for m, r in a.iterrows():
    plt.annotate(LABEL[m], (r['any_violation'] * 100, r['rmse']), fontsize=8,
                 xytext=(4, 2), textcoords='offset points')
plt.xlabel('grid points violating butterfly or calendar condition (%)')
plt.ylabel('mean RMSE, random mask (vol points)')
plt.title('Accuracy vs arbitrage (lower-left is better)'); plt.show()

## 6. Sparsity: how fast does each model degrade as more quotes are hidden?

In [ ]:
sp = pd.read_csv('../results/sparsity.csv')
curve = sp.groupby(['model', 'frac'])['rmse'].mean().unstack('model')
curve.rename(columns=LABEL).plot(marker='o', figsize=(8, 5), logy=True)
plt.xlabel('share of quotes hidden'); plt.ylabel('mean RMSE (vol points, log scale)')
plt.title('Sparsity sweep (every 10th test day)'); plt.show()
curve.rename(columns=LABEL).round(2)

## 7. Hedging: does a better surface give better deltas?
For each hidden option: one-day delta hedge using the model's IV vs using the market's IV.
Error in dollars per option (lower is better).

In [ ]:
data = pd.read_parquet('../data/processed/options_features.parquet',
                       columns=['quote_date', 'expire_date', 'strike', 'type', 'mid', 'F',
                                'underlying_last'])
hedge = {}
for m in [m for m in ['linear', 'svi', 'ssvi', 'xgb_svi', 'gp', 'nn'] if m in MODELS]:
    p = pd.read_parquet(f'../results/{m}/preds.parquet')
    hedge[m] = hedge_summary(hedge_errors(p, data))
pd.concat(hedge).rename(index=LABEL).round(4)

## 8. A crisis day up close

In [ ]:
DAY = pd.Timestamp('2020-03-16')
m_show = [m for m in ['svi', 'xgb_svi', 'nn'] if m in MODELS]
preds = {m: pd.read_parquet(f'../results/{m}/preds.parquet',
                            filters=[('date', '==', DAY), ('mask', '==', 'wing')]) for m in m_show}
p0 = preds[m_show[0]]
exp = np.sort(p0['expire_date'].unique())
picks = exp[np.linspace(0, len(exp) - 1, 3).astype(int)]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, e in zip(axes, picks):
    for m in m_show:
        q = preds[m][preds[m]['expire_date'] == e].sort_values('k')
        ax.plot(q['k'], q['iv_pred'], 'o-', ms=3, label=LABEL[m])
    q = p0[p0['expire_date'] == e]
    ax.scatter(q['k'], q['iv_true'], c='black', s=20, marker='x', label='market (hidden)')
    ax.set_title(f'{DAY.date()} wing mask, expiry {pd.Timestamp(e).date()}'); ax.set_xlabel('k')
axes[0].set_ylabel('implied volatility'); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 9. Conclusions
*Write the answers here once the numbers above are final:*
1. Which model is most accurate on each mask, and by how much versus SVI (with the CI and p-value)?
2. Does the ranking change in the COVID crash and the 2022 bear market?
3. Which regions (wings, short maturities) drive the errors?
4. Is there an accuracy–arbitrage trade-off (SSVI vs SVI vs NN with/without penalty)?
5. Does better accuracy translate into better hedging?
6. Limitations: American options (late-2022 put/call IV gap), provider IVs, one underlying,
   three turbulent years.